In [1]:
import radiate as rd
import polars as pl
from IPython.display import display, HTML

rd.random.seed(67123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [ ]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.linear()],
        edge=rd.Op.weight(),
        output=rd.Op.linear(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .alters(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-06-03T18:19:27.722931Z  INFO Epoch 1    | Score:   2.0038 | Time: 2.50ms
2026-06-03T18:19:27.723074Z  INFO Epoch 2    | Score:   2.0038 | Time: 2.57ms
2026-06-03T18:19:27.723160Z  INFO Epoch 3    | Score:   1.7747 | Time: 2.63ms
2026-06-03T18:19:27.723243Z  INFO Epoch 4    | Score:   1.7433 | Time: 2.68ms
2026-06-03T18:19:27.723332Z  INFO Epoch 5    | Score:   1.6821 | Time: 2.74ms
2026-06-03T18:19:27.723424Z  INFO Epoch 6    | Score:   1.6821 | Time: 2.81ms
2026-06-03T18:19:27.723524Z  INFO Epoch 7    | Score:   1.6821 | Time: 2.88ms
2026-06-03T18:19:27.723608Z  INFO Epoch 8    | Score:   1.6821 | Time: 2.93ms
2026-06-03T18:19:27.723688Z  INFO Epoch 9    | Score:   1.6821 | Time: 2.98ms
2026-06-03T18:19:27.723763Z  INFO Epoch 10   | Score:   1.6821 | Time: 3.03ms
2026-06-03T18:19:27.723847Z  INFO Epoch 11   | Score:   1.6821 | Time: 3.09ms
2026-06-03T18:19:27.723929Z  INFO Epoch 12   | Score:   1.6821 | Time: 3.14ms
2026-06-03T18:19:27.724002Z  INFO Epoch 13   | Score:   1.6821 |

In [4]:
# collector.plot()
print(result.metrics().dashboard())

[35 metrics, 192445 updates]
----- Metrics ----- (35 :: 192445) 
  carryover: 0.170  diversity: 0.667  unique_members: 71  unique_scores: 66  improvements: 531  iter_time(mean): 408.290µs

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt       | Entr      
-------------------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 0.810      | 0.000      | 3.000      | 100    | 0.081        | 1.089      | 13.730     | 93.994     | -         
scores                     | dist   | 12.137     | 0.002      | 514.158    | 100    | 1.214        | 64.526     | 6.412      | 46.553     | -         
size.genome                | dist   | 39.550     | 39.000     | 41.000     | 100    | 3.955        | 0.557      | 0.000      | 0.000      | -         


== Statistics ==
Name                  

In [5]:
eval_results = result.value().eval(inputs)
accuracy = rd.accuracy(result.value(), inputs, answers, loss=rd.MSE)
accuracy

PyAccuracy("Regression Accuracy" {
	N: 20 
	Accuracy: 99.60%
	R² Score: 0.99962
	RMSE: 0.03941
	Loss (MSE): 0.00155
})

In [6]:
df = collector.to_polars(lazy=False)
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",8.0,108.0,54.0,65.053825,4232.0,NaN,8.0,100.0,2,null,null,null,null,null,null,0,2,"[""statistic""]"
"""step.evaluate.time""",0.000537,0.001196,0.000598,0.000087,7.5134e-9,NaN,0.000537,0.000659,2,1196µs,598µs,86µs,536µs,659µs,0µs,0,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,20.0,20.0,0.0,0.0,NaN,20.0,20.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000221,0.000221,0.000221,0.0,0.0,NaN,0.000221,0.000221,1,220µs,220µs,0µs,220µs,220µs,0µs,0,1,"[""selector"", ""time""]"
"""selector.tournament""",80.0,80.0,80.0,0.0,0.0,NaN,80.0,80.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""step.audit.time""",0.000008,0.008529,0.000009,0.000003,8.2521e-12,0.0,0.000007,0.00009,1000,8529µs,8µs,2µs,6µs,90µs,0µs,999,1,"[""time"", ""step""]"
"""time""",0.000555,0.40829,0.000408,0.000134,1.8045e-8,0.0,0.000036,0.002498,1000,408290µs,408µs,134µs,36µs,2498µs,0µs,999,1,"[""time""]"
"""index""",1000.0,1000.0,1000.0,0.0,0.0,NaN,1000.0,1000.0,1,null,null,null,null,null,null,0,1,"[""statistic""]"


In [7]:
filtered = (
    df.filter(pl.col("name") == "score.improvement")
    .select("generation")
    .unique()
    .sort("generation")
)
filtered

generation
i64
0
2
3
4
24
…
986
992
995


In [8]:
display(HTML(filtered._repr_html_()))

generation
i64
0
2
3
4
24
…
986
992
995
